In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_fcaf_probere1 import ViViTBackbone, ViViTFCAFLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-08 15:44:48,397]::2730006395::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# ViViT-FCAF Block-wise Linear Probe
# ============================================================
#
# Probe targets:
#   block1 ... block6 : pure ViViT representations
#   final_cross       : representation after the single
#                       final-stage cross-attention
#
# The trained backbone is frozen. Only probe heads are optimized.
# ============================================================

PROBE_SEED = 42
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 50
patience = 10

# Set this to the checkpoint produced by the corresponding
# trained ViViT-FCAF model.
BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(PROBE_SEED)

g = torch.Generator()
g.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker,
    generator=g
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker
)

# ------------------------------------------------------------
# 1. Build the same trained ViViT-FCAF backbone
# ------------------------------------------------------------
backbone = ViViTBackbone(
    2, 14, 14, 2, 2, 2, 5, 96, 6, 8, 384, model=3
).to(device)

ckpt = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

state_dict = (
    ckpt["model"]
    if isinstance(ckpt, dict) and "model" in ckpt
    else ckpt
)

backbone.load_state_dict(
    state_dict,
    strict=True
)

# ------------------------------------------------------------
# 2. Frozen backbone + independent linear probes
# ------------------------------------------------------------
model = ViViTFCAFLinearProbe(
    backbone=backbone,
    dim=96,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")

assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.L1Loss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)

probe_names = list(model.probe_heads.keys())

best_mae = {
    name: float("inf")
    for name in probe_names
}

best_epoch = {
    name: -1
    for name in probe_names
}

best_mean_val_mae = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):

    # ========================================================
    # Train probes only
    # ========================================================
    model.train()

    train_sum = {
        name: 0.0
        for name in probe_names
    }

    for audiodata, fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
        audiodata = audiodata.to(device)

        optimizer.zero_grad()

        outputs = model(
            fullshot,
            audiodata
        )

        losses = {
            name: criterion(
                outputs[name],
                big_five_data
            )
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        for name in probe_names:
            train_sum[name] += losses[name].item()

    train_mae = {
        name: train_sum[name] / len(train_dataloader)
        for name in probe_names
    }

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_abs_sum = {
        name: 0.0
        for name in probe_names
    }

    val_count = 0

    with torch.no_grad():

        for audiodata, fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
            audiodata = audiodata.to(device)

            outputs = model(
                fullshot,
                audiodata
            )

            val_count += big_five_data.numel()

            for name in probe_names:
                val_abs_sum[name] += torch.abs(
                    outputs[name] - big_five_data
                ).sum().item()

    val_mae = {
        name: val_abs_sum[name] / val_count
        for name in probe_names
    }

    # Each probe keeps its own best epoch.
    for name in probe_names:
        if val_mae[name] < best_mae[name]:
            best_mae[name] = val_mae[name]
            best_epoch[name] = epoch + 1

    mean_val_mae = float(
        np.mean(
            [val_mae[name] for name in probe_names]
        )
    )

    if mean_val_mae < best_mean_val_mae:
        best_mean_val_mae = mean_val_mae
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 80)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MAE={mean_val_mae:.6f} | "
        f"Patience={no_improve_count}/{patience}"
    )

    for name in probe_names:
        print(
            f"{name:<12} | "
            f"train MAE {train_mae[name]:.6f} | "
            f"val MAE {val_mae[name]:.6f} | "
            f"1-MAE {1.0 - val_mae[name]:.6f} | "
            f"best 1-MAE {1.0 - best_mae[name]:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch + 1}. "
            f"Best mean validation MAE={best_mean_val_mae:.6f}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary
# ============================================================

print("\n" + "=" * 80)
print("FINAL ViViT-FCAF LINEAR PROBE RESULTS")
print("=" * 80)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'MAE':>12}"
    f"{'1-MAE':>12}"
)
print("-" * 80)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mae[name]:>12.6f}"
        f"{1.0 - best_mae[name]:>12.6f}"
    )

print("=" * 80)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable parameters: 0
Probe trainable parameters:    4739


Probe Train 1/50: 100%|█████████▉| 1497/1498 [03:17<00:00,  8.36it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Probe Valid 1/50: 100%|██████████| 498/498 [01:10<00:00,  7.09it/s]



--------------------------------------------------------------------------------
Epoch 001 | Mean Val MAE=0.085509 | Patience=0/10
block1       | train MAE 0.043021 | val MAE 0.088058 | 1-MAE 0.911942 | best 1-MAE 0.911942
block2       | train MAE 0.044384 | val MAE 0.085653 | 1-MAE 0.914347 | best 1-MAE 0.914347
block3       | train MAE 0.041092 | val MAE 0.086482 | 1-MAE 0.913518 | best 1-MAE 0.913518
block4       | train MAE 0.043037 | val MAE 0.086562 | 1-MAE 0.913438 | best 1-MAE 0.913438
block5       | train MAE 0.038962 | val MAE 0.086029 | 1-MAE 0.913971 | best 1-MAE 0.913971
block6       | train MAE 0.042075 | val MAE 0.085023 | 1-MAE 0.914977 | best 1-MAE 0.914977
final_cross  | train MAE 0.030506 | val MAE 0.080753 | 1-MAE 0.919247 | best 1-MAE 0.919247


Probe Valid 2/50: 100%|██████████| 498/498 [01:11<00:00,  7.01it/s]



--------------------------------------------------------------------------------
Epoch 002 | Mean Val MAE=0.085794 | Patience=1/10
block1       | train MAE 0.036059 | val MAE 0.087048 | 1-MAE 0.912952 | best 1-MAE 0.912952
block2       | train MAE 0.035559 | val MAE 0.086467 | 1-MAE 0.913533 | best 1-MAE 0.914347
block3       | train MAE 0.034860 | val MAE 0.086336 | 1-MAE 0.913664 | best 1-MAE 0.913664
block4       | train MAE 0.034574 | val MAE 0.087155 | 1-MAE 0.912845 | best 1-MAE 0.913438
block5       | train MAE 0.034195 | val MAE 0.085450 | 1-MAE 0.914550 | best 1-MAE 0.914550
block6       | train MAE 0.033780 | val MAE 0.085929 | 1-MAE 0.914071 | best 1-MAE 0.914977
final_cross  | train MAE 0.019848 | val MAE 0.082175 | 1-MAE 0.917825 | best 1-MAE 0.919247


...


--------------------------------------------------------------------------------
Epoch 037 | Mean Val MAE=0.083993 | Patience=8/10
block1       | train MAE 0.029042 | val MAE 0.084471 | 1-MAE 0.915529 | best 1-MAE 0.915730
block2       | train MAE 0.028866 | val MAE 0.084689 | 1-MAE 0.915311 | best 1-MAE 0.915715
block3       | train MAE 0.028279 | val MAE 0.084535 | 1-MAE 0.915465 | best 1-MAE 0.915793
block4       | train MAE 0.027745 | val MAE 0.084471 | 1-MAE 0.915529 | best 1-MAE 0.915902
block5       | train MAE 0.027612 | val MAE 0.084433 | 1-MAE 0.915567 | best 1-MAE 0.915886
block6       | train MAE 0.027497 | val MAE 0.084332 | 1-MAE 0.915668 | best 1-MAE 0.915907
final_cross  | train MAE 0.016420 | val MAE 0.081018 | 1-MAE 0.918982 | best 1-MAE 0.919379


Probe Valid 38/50: 100%|██████████| 498/498 [00:46<00:00, 10.67it/s]



--------------------------------------------------------------------------------
Epoch 038 | Mean Val MAE=0.084129 | Patience=9/10
block1       | train MAE 0.028968 | val MAE 0.084627 | 1-MAE 0.915373 | best 1-MAE 0.915730
block2       | train MAE 0.028647 | val MAE 0.084981 | 1-MAE 0.915019 | best 1-MAE 0.915715
block3       | train MAE 0.028037 | val MAE 0.084775 | 1-MAE 0.915225 | best 1-MAE 0.915793
block4       | train MAE 0.027680 | val MAE 0.084608 | 1-MAE 0.915392 | best 1-MAE 0.915902
block5       | train MAE 0.027442 | val MAE 0.084572 | 1-MAE 0.915428 | best 1-MAE 0.915886
block6       | train MAE 0.027319 | val MAE 0.084521 | 1-MAE 0.915479 | best 1-MAE 0.915907
final_cross  | train MAE 0.016551 | val MAE 0.080820 | 1-MAE 0.919180 | best 1-MAE 0.919379


Probe Valid 39/50: 100%|██████████| 498/498 [00:46<00:00, 10.66it/s]


--------------------------------------------------------------------------------
Epoch 039 | Mean Val MAE=0.084658 | Patience=10/10
block1       | train MAE 0.029008 | val MAE 0.084864 | 1-MAE 0.915136 | best 1-MAE 0.915730
block2       | train MAE 0.028819 | val MAE 0.085701 | 1-MAE 0.914299 | best 1-MAE 0.915715
block3       | train MAE 0.028248 | val MAE 0.085011 | 1-MAE 0.914989 | best 1-MAE 0.915793
block4       | train MAE 0.027816 | val MAE 0.085069 | 1-MAE 0.914931 | best 1-MAE 0.915902
block5       | train MAE 0.027604 | val MAE 0.085457 | 1-MAE 0.914543 | best 1-MAE 0.915886
block6       | train MAE 0.027411 | val MAE 0.085354 | 1-MAE 0.914646 | best 1-MAE 0.915907
final_cross  | train MAE 0.016388 | val MAE 0.081152 | 1-MAE 0.918848 | best 1-MAE 0.919379

Early stopping at epoch 39. Best mean validation MAE=0.083722

FINAL ViViT-FCAF LINEAR PROBE RESULTS
Representation      Best Epoch         MAE       1-MAE
------------------------------------------------------------------